### Data Ingestion


In [2]:
### document structure
from langchain_core.documents import Document
doc = Document(
    page_content="This is a main content i am using to create a RAG",
    metadata={"source": "exapmle.com", "author": "John Doe", "date": "2024-06-01"
    }
)
doc



Document(metadata={'source': 'exapmle.com', 'author': 'John Doe', 'date': '2024-06-01'}, page_content='This is a main content i am using to create a RAG')

In [ ]:
## create a simple txt file

import os
os.makedirs("../data/text_files", exist_ok=True)

In [ ]:
sample_text={
    "../data/text_files/python_intro.txt": """Machine Learnign basics:
    Machine learning is a subset of artificial intelligence where algorithms analyze data to learn patterns and make predictions or decisions without
    being explicitly programmed for every task.  Unlike traditional programming, which relies on fixed rules, machine learning models improve their accuracy over time as they process more experience (data). 

    Supervised Learning: Models are trained on labeled data (data with known answers) to predict outcomes for new inputs, commonly used for classification and regression tasks like spam detection or price forecasting. 
    Unsupervised Learning: Algorithms analyze unlabeled data to discover hidden patterns, structures, or groupings, such as clustering similar customer behaviors or reducing data dimensions. 
    Reinforcement Learning: Agents learn to make sequences of decisions by receiving rewards or penalties for their actions, aiming to maximize cumulative reward in dynamic environments like gaming or robotics.
       
    """
}

for filepath, content in sample_text.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("Sample text files created successfully.")

In [4]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
docs = loader.load()
print(docs)




[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Machine Learnign basics:\n    Machine learning is a subset of artificial intelligence where algorithms analyze data to learn patterns and make predictions or decisions without\n    being explicitly programmed for every task.  Unlike traditional programming, which relies on fixed rules, machine learning models improve their accuracy over time as they process more experience (data). \n\n    Supervised Learning: Models are trained on labeled data (data with known answers) to predict outcomes for new inputs, commonly used for classification and regression tasks like spam detection or price forecasting. \n    Unsupervised Learning: Algorithms analyze unlabeled data to discover hidden patterns, structures, or groupings, such as clustering similar customer behaviors or reducing data dimensions. \n    Reinforcement Learning: Agents learn to make sequences of decisions by receiving rewards or penalties for their

In [8]:
### directory Loader
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader = DirectoryLoader(
    "../data/text_files", 
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=False
)

documets = dir_loader.load()
documets


[Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Machine Learnign basics:\n    Machine learning is a subset of artificial intelligence where algorithms analyze data to learn patterns and make predictions or decisions without\n    being explicitly programmed for every task.  Unlike traditional programming, which relies on fixed rules, machine learning models improve their accuracy over time as they process more experience (data). \n\n    Supervised Learning: Models are trained on labeled data (data with known answers) to predict outcomes for new inputs, commonly used for classification and regression tasks like spam detection or price forecasting. \n    Unsupervised Learning: Algorithms analyze unlabeled data to discover hidden patterns, structures, or groupings, such as clustering similar customer behaviors or reducing data dimensions. \n    Reinforcement Learning: Agents learn to make sequences of decisions by receiving rewards or penalties for th

###Embadings and chunckings

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Tuple
from sklearn.metrics.pairwise import cosine_similarity



c:\Users\Niraj\Desktop\udemy-agentic-ai\YTRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
class EmbadingManager:
    """handles documets embading generation using sentenceTrasnformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the EmbadingManager with a specified model name.
        
        Args:
            model_name (str): The name of the SentenceTransformer model to use for embedding generation.
        """
        self.model = SentenceTransformer(model_name)
        self.model = None
        self.load_model()

    def load_model(self):        
        """Load the SentenceTransformer model."""
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Model '{self.model_name}' loaded successfully.")
        except Exception as e:
            print(f"Error occurred while loading the model: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts.
        
        Args:
            texts (List[str]): A list of strings for which to generate embeddings.
        
        Returns:
            np.ndarray: An array of embeddings.
        """
        if self.model is None:
            raise ValueError("Model not loaded. Please load the model first.")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts,show_progress_bar=True)
        print(f"Embeddings generated for {embeddings.shape} texts.")
        return embeddings